In [1]:
import pandas as pd

# -----------------------------
# Load source datasets (read-only)
# -----------------------------
df_vanilla = pd.read_parquet("../results1/unified_baseline/gaia/vanilla/gpt_4o_mini/responses.parquet")
df_zero_shot_cot = pd.read_parquet("../results1/unified_baseline/gaia/zero_shot_cot/gpt_4o_mini/responses.parquet")
df_processed = pd.read_parquet("../datasets/processed/gaia.parquet")

# -----------------------------
# Step 1: initial complexity level from vanilla + cot
# Rules:
# both False -> 2
# vanilla True (including both True) -> 0
# cot True and vanilla False -> 1
# anything else -> 2
# -----------------------------
complexity_df = (
    df_vanilla[["query_id", "dataset", "model", "query", "ground_truth", "predicted_answer", "is_correct"]]
    .rename(
        columns={
            "predicted_answer": "predicted_vanilla",
            "is_correct": "is_correct_vanilla",
        }
    )
    .merge(
        df_zero_shot_cot[["query_id", "predicted_answer", "is_correct"]].rename(
            columns={
                "predicted_answer": "predicted_cot",
                "is_correct": "is_correct_cot",
            }
        ),
        on="query_id",
        how="inner",
    )
)

complexity_df["complexity_ground_truth"] = 2

mask_level_0 = complexity_df["is_correct_vanilla"] == True
mask_level_1 = (
    (complexity_df["is_correct_cot"] == True)
    & (complexity_df["is_correct_vanilla"] == False)
)

complexity_df.loc[mask_level_0, "complexity_ground_truth"] = 0
complexity_df.loc[mask_level_1, "complexity_ground_truth"] = 1



In [2]:
df_processed['steps_num']>9 

0       True
1       True
2      False
3      False
4      False
       ...  
160     True
161     True
162     True
163     True
164    False
Name: steps_num, Length: 165, dtype: boolean

In [3]:
# Corrected pipeline: cast processed numeric fields before threshold comparisons
import pandas as pd

df_vanilla = pd.read_parquet("../results1/unified_baseline/gaia/vanilla/gpt_4o_mini/responses.parquet")
df_zero_shot_cot = pd.read_parquet("../results1/unified_baseline/gaia/zero_shot_cot/gpt_4o_mini/responses.parquet")
df_processed = pd.read_parquet("../datasets/processed/gaia.parquet")

# Step 1
complexity_df = (
    df_vanilla[["query_id", "dataset", "model", "query", "ground_truth", "predicted_answer", "is_correct"]]
    .rename(columns={"predicted_answer": "predicted_vanilla", "is_correct": "is_correct_vanilla"})
    .merge(
        df_zero_shot_cot[["query_id", "predicted_answer", "is_correct"]].rename(
            columns={"predicted_answer": "predicted_cot", "is_correct": "is_correct_cot"}
        ),
        on="query_id",
        how="inner",
    )
)
complexity_df["complexity_ground_truth"] = 2
complexity_df.loc[complexity_df["is_correct_vanilla"] == True, "complexity_ground_truth"] = 0
complexity_df.loc[
    (complexity_df["is_correct_cot"] == True) & (complexity_df["is_correct_vanilla"] == False),
    "complexity_ground_truth",
] = 1

# Step 2 (only rows at level 2)
processed_cols = ["id", "query", "answer", "level", "annotator_steps", "annotator_tools", "file_name", "steps_num", "tool_num", "time_taken"]
df_processed_view = df_processed[processed_cols].copy()

level2_rows = complexity_df[complexity_df["complexity_ground_truth"] == 2].copy()
level2_enriched = level2_rows.merge(
    df_processed_view.rename(columns={"id": "processed_id", "answer": "processed_answer"}),
    on="query",
    how="left",
)

# FIX: coerce possible string columns to numeric
for col in ["tool_num", "time_taken", "annotator_steps"]:
    level2_enriched[col] = pd.to_numeric(level2_enriched[col], errors="coerce")

print(level2_enriched)

mask_keep_2 = (
    (level2_enriched["tool_num"] > 1)
    & (level2_enriched["tool_num"] < 3)
    & (level2_enriched["time_taken"] < 20)
)
mask_to_3 = (level2_enriched["steps_num"] > 8) & (level2_enriched["time_taken"] > 20)

level2_enriched.loc[mask_keep_2, "complexity_ground_truth"] = 2
level2_enriched.loc[mask_to_3, "complexity_ground_truth"] = 3

final_complexity_df = complexity_df.merge(
    level2_enriched[["query_id", "complexity_ground_truth"]].rename(
        columns={"complexity_ground_truth": "complexity_ground_truth_refined"}
    ),
    on="query_id",
    how="left",
)
final_complexity_df["complexity_ground_truth"] = final_complexity_df[
    "complexity_ground_truth_refined"
].combine_first(final_complexity_df["complexity_ground_truth"])
final_complexity_df = final_complexity_df.drop(columns=["complexity_ground_truth_refined"])

print("final rows:", len(final_complexity_df))
print(final_complexity_df["complexity_ground_truth"].value_counts(dropna=False).sort_index())
final_complexity_df.head()


                                 query_id dataset        model  \
0    17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc    gaia  gpt-4o-mini   
1    04a04a9b-226c-43fd-b319-d5e89743676f    gaia  gpt-4o-mini   
2    14569e28-c88c-43e4-8c32-097d35b9a67d    gaia  gpt-4o-mini   
3    e1fc63a2-da7a-432f-be78-7c4a95598703    gaia  gpt-4o-mini   
4    32102e3e-d12a-4209-9163-7b3a104efe5d    gaia  gpt-4o-mini   
..                                    ...     ...          ...   
141  0512426f-4d28-49f0-be77-06d05daec096    gaia  gpt-4o-mini   
142  0bdb7c40-671d-4ad1-9ce3-986b159c0ddc    gaia  gpt-4o-mini   
143  08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715    gaia  gpt-4o-mini   
144  853c8244-429e-46ca-89f2-addf40dfb2bd    gaia  gpt-4o-mini   
145  7a4a336d-dcfa-45a0-b014-824c7619e8de    gaia  gpt-4o-mini   

                                                 query  \
0    I’m researching species that became invasive a...   
1    If we assume all articles published by Nature ...   
2    In Unlambda, what exact char

,query_id,dataset,model,query,ground_truth,predicted_vanilla,is_correct_vanilla,predicted_cot,is_correct_cot,complexity_ground_truth
0,c61d22de-5f6c-4958-a7f6-5e9707bd3466,gaia,gpt-4o-mini,A paper about AI regulation that was originall...,egalitarian,egalitarian,True,market,False,0.0
1,17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc,gaia,gpt-4o-mini,I’m researching species that became invasive a...,34689,"33139, 33140, 33141, 33142, 33143",False,"33139, 00601",False,2.0
2,04a04a9b-226c-43fd-b319-d5e89743676f,gaia,gpt-4o-mini,If we assume all articles published by Nature ...,41,"1,000",False,50,False,2.0
3,14569e28-c88c-43e4-8c32-097d35b9a67d,gaia,gpt-4o-mini,"In Unlambda, what exact charcter or text needs...",backtick,space,False,"Therefore, the character needed is a space",False,2.0
4,e1fc63a2-da7a-432f-be78-7c4a95598703,gaia,gpt-4o-mini,If Eliud Kipchoge could maintain his record-ma...,17,1000,False,1000,False,2.0


In [4]:
# Show only level 3 rows
level_3_df = final_complexity_df[final_complexity_df["complexity_ground_truth"] == 3]
print("level 3 rows:", len(level_3_df))
level_3_df


level 3 rows: 5


,query_id,dataset,model,query,ground_truth,predicted_vanilla,is_correct_vanilla,predicted_cot,is_correct_cot,complexity_ground_truth
23,f0f46385-fc03-4599-b5d3-f56496c3e69f,gaia,gpt-4o-mini,In terms of geographical distance between capi...,"Indonesia, Myanmar","Brunei, Vietnam",False,"Brunei, Vietnam",False,3.0
31,983bba7c-c092-455f-b6c9-7857003d48fc,gaia,gpt-4o-mini,What animals that were mentioned in both Ilias...,mice,fish,False,unknown,False,3.0
62,56db2318-640f-477a-a82f-bc93ad13e882,gaia,gpt-4o-mini,The following numbers function similarly to IS...,"7, 9","3, 4",False,"2, 3",False,3.0
95,851e570a-e3de-4d84-bcfa-cc85578baa59,gaia,gpt-4o-mini,I thought we could try a fun word puzzle toget...,Briniest,repetition,False,BRAIN,False,3.0
109,c3a79cfe-8206-451f-aca8-3fec8ebe51d3,gaia,gpt-4o-mini,The year is 2022. I am at the National Air and...,8,3,False,3,False,3.0


In [5]:
# Show steps, time taken, and tools used (for level 3 rows)
level_3_details = level_3_df.merge(
    df_processed[["query", "steps_num", "time_taken", "tool_num", "annotator_tools"]],
    on="query",
    how="left",
)

level_3_details[
    [
        "query_id",
        "query",
        "complexity_ground_truth",
        "steps_num",
        "time_taken",
        "tool_num",
        "annotator_tools",
    ]
]


,query_id,query,complexity_ground_truth,steps_num,time_taken,tool_num,annotator_tools
0,f0f46385-fc03-4599-b5d3-f56496c3e69f,In terms of geographical distance between capi...,3.0,13,45,3,1. Search engine\n2. Web browser\n3. Microsoft...
1,983bba7c-c092-455f-b6c9-7857003d48fc,What animals that were mentioned in both Ilias...,3.0,14,25,3,1. Web browser\n2. Search engine\n3. PDF access
2,56db2318-640f-477a-a82f-bc93ad13e882,The following numbers function similarly to IS...,3.0,32,60,1,1. a calculator
3,851e570a-e3de-4d84-bcfa-cc85578baa59,I thought we could try a fun word puzzle toget...,3.0,12,40,4,1. A file interface\n2. A Python IDE\n3. A web...
4,c3a79cfe-8206-451f-aca8-3fec8ebe51d3,The year is 2022. I am at the National Air and...,3.0,20,50,4,1. Web Browser\n2. Search Engine\n3. Access to...


# Show only level 3 rows
level_3_df = final_complexity_df[final_complexity_df["complexity_ground_truth"] == 3]
print("level 3 rows:", len(level_3_df))
level_3_df



In [10]:
df_vanilla


,query_id,dataset,modality,model,query,ground_truth,predicted_answer,is_correct,raw_model_output,reasoning_trace,...,completion_tokens,total_tokens,cost_usd,latency_s,error_type,error_message,system_prompt_version,user_prompt_version,prompt_hash,metadata
0,c61d22de-5f6c-4958-a7f6-5e9707bd3466,gaia,vanilla,gpt-4o-mini,A paper about AI regulation that was originall...,egalitarian,egalitarian,True,egalitarian,NaN,...,2,242,0.000037,1.183,None,None,v1,v1,caa1388609f92a3bef940bc304c96482d5275dce578317...,"{'file_name': '', 'level': '2'}"
1,17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc,gaia,vanilla,gpt-4o-mini,I’m researching species that became invasive a...,34689,"33139, 33140, 33141, 33142, 33143",False,"33139, 33140, 33141, 33142, 33143",NaN,...,18,284,0.000051,0.907,None,None,v1,v1,68169032322546ce06016745f9d3c86720df4bab0131ae...,"{'file_name': '', 'level': '2'}"
2,04a04a9b-226c-43fd-b319-d5e89743676f,gaia,vanilla,gpt-4o-mini,If we assume all articles published by Nature ...,41,"1,000",False,"1,000",NaN,...,3,244,0.000038,0.468,None,None,v1,v1,16395f8760094e68133074265c2ac5320dd55402054f7b...,"{'file_name': '', 'level': '2'}"
3,14569e28-c88c-43e4-8c32-097d35b9a67d,gaia,vanilla,gpt-4o-mini,"In Unlambda, what exact charcter or text needs...",backtick,space,False,space,NaN,...,1,257,0.000039,0.681,None,None,v1,v1,8a19c4c9d09cdc62d62ccdcb5c0b5c863a73f0542dea02...,"{'file_name': '', 'level': '2'}"
4,e1fc63a2-da7a-432f-be78-7c4a95598703,gaia,vanilla,gpt-4o-mini,If Eliud Kipchoge could maintain his record-ma...,17,1000,False,1000,NaN,...,2,250,0.000038,0.522,None,None,v1,v1,2b41c64683ac9354db7205f355cb843acaa9b08351b87d...,"{'file_name': '', 'level': '1'}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,0bdb7c40-671d-4ad1-9ce3-986b159c0ddc,gaia,vanilla,gpt-4o-mini,In NASA's Astronomy Picture of the Day on 2006...,White; 5876,Cernan; 0,False,Cernan; 0,NaN,...,6,289,0.000046,0.608,None,None,v1,v1,d16f796f1bb7cb0a59fd3a470330a36c626e6b88d020a9...,"{'file_name': '', 'level': '3'}"
161,08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715,gaia,vanilla,gpt-4o-mini,"In the film Goldfinger, what color was the obj...","orange, white",gold,False,gold,NaN,...,1,217,0.000033,0.479,None,None,v1,v1,a68a6734024c75059676a7068887d2c97000498a326a69...,"{'file_name': '', 'level': '2'}"
162,db4fd70a-2d37-40ea-873f-9433dc5e301f,gaia,vanilla,gpt-4o-mini,"As of May 2023, how many stops are between Sou...",10,6,False,6,NaN,...,1,201,0.000031,0.516,None,None,v1,v1,141905b360508a17c41ddf91e01904c138fc2b56c806d6...,"{'file_name': '', 'level': '2'}"
163,853c8244-429e-46ca-89f2-addf40dfb2bd,gaia,vanilla,gpt-4o-mini,In the 2015 Metropolitan Museum of Art exhibit...,11,9,False,9,NaN,...,1,210,0.000032,0.628,None,None,v1,v1,f1d23da0f5e410c41dbb5f247bff60d5c25b8f460a3273...,"{'file_name': '', 'level': '2'}"


In [6]:
# Updated version of the same pipeline block (with tool-based level 3)
import pandas as pd

df_vanilla = pd.read_parquet("../results1/unified_baseline/gaia/vanilla/gpt_4o_mini/responses.parquet")
df_zero_shot_cot = pd.read_parquet("../results1/unified_baseline/gaia/zero_shot_cot/gpt_4o_mini/responses.parquet")
df_processed = pd.read_parquet("../datasets/processed/gaia.parquet")

# Step 1
complexity_df = (
    df_vanilla[["query_id", "dataset", "model", "query", "ground_truth", "predicted_answer", "is_correct"]]
    .rename(columns={"predicted_answer": "predicted_vanilla", "is_correct": "is_correct_vanilla"})
    .merge(
        df_zero_shot_cot[["query_id", "predicted_answer", "is_correct"]].rename(
            columns={"predicted_answer": "predicted_cot", "is_correct": "is_correct_cot"}
        ),
        on="query_id",
        how="inner",
    )
)
complexity_df["complexity_ground_truth"] = 2
complexity_df.loc[complexity_df["is_correct_vanilla"] == True, "complexity_ground_truth"] = 0
complexity_df.loc[
    (complexity_df["is_correct_cot"] == True) & (complexity_df["is_correct_vanilla"] == False),
    "complexity_ground_truth",
] = 1

# Step 2 (only rows at level 2)
processed_cols = ["id", "query", "answer", "level", "annotator_steps", "annotator_tools", "file_name", "steps_num", "tool_num", "time_taken"]
df_processed_view = df_processed[processed_cols].copy()

level2_rows = complexity_df[complexity_df["complexity_ground_truth"] == 2].copy()
level2_enriched = level2_rows.merge(
    df_processed_view.rename(columns={"id": "processed_id", "answer": "processed_answer"}),
    on="query",
    how="left",
)

for col in ["tool_num", "time_taken", "annotator_steps", "steps_num", "level"]:
    level2_enriched[col] = pd.to_numeric(level2_enriched[col], errors="coerce")

# Normalize tool list strings like:
# '1. Web browser\n2. Search engine\n3. Image processing tools\n4. Calculator'
def _normalize_tools(tool_blob: object) -> list[str]:
    if pd.isna(tool_blob):
        return []
    tools: list[str] = []
    for raw_line in str(tool_blob).splitlines():
        line = raw_line.strip().lower()
        if not line:
            continue
        # Remove leading numbering such as '1. ' / '2) '
        line = line.lstrip("0123456789").lstrip(".) ")
        # Collapse repeated spaces
        line = " ".join(line.split())
        if line:
            tools.append(line)
    return tools

normalized_tool_lists = level2_enriched["annotator_tools"].apply(_normalize_tools)

mask_tool_based_3 = normalized_tool_lists.apply(
    lambda tools: ("python ide" in tools) or ("file inference" in tools)
)

mask_keep_2 = (
    (level2_enriched["tool_num"] > 1)
    & (level2_enriched["tool_num"] < 3)
    & (level2_enriched["time_taken"] < 20)
)
mask_to_3 = (
    ((level2_enriched["steps_num"] > 8) & (level2_enriched["time_taken"] > 25))
)

level2_enriched.loc[mask_keep_2, "complexity_ground_truth"] = 2
level2_enriched.loc[mask_to_3, "complexity_ground_truth"] = 3

final_complexity_df = complexity_df.merge(
    level2_enriched[["query_id", "complexity_ground_truth"]].rename(
        columns={"complexity_ground_truth": "complexity_ground_truth_refined"}
    ),
    on="query_id",
    how="left",
)
final_complexity_df["complexity_ground_truth"] = final_complexity_df[
    "complexity_ground_truth_refined"
].combine_first(final_complexity_df["complexity_ground_truth"])
final_complexity_df = final_complexity_df.drop(columns=["complexity_ground_truth_refined"])

print("final rows:", len(final_complexity_df))
print(final_complexity_df["complexity_ground_truth"].value_counts(dropna=False).sort_index())
final_complexity_df.head()


final rows: 165
complexity_ground_truth
0.0      7
1.0     12
2.0    142
3.0      4
Name: count, dtype: int64


,query_id,dataset,model,query,ground_truth,predicted_vanilla,is_correct_vanilla,predicted_cot,is_correct_cot,complexity_ground_truth
0,c61d22de-5f6c-4958-a7f6-5e9707bd3466,gaia,gpt-4o-mini,A paper about AI regulation that was originall...,egalitarian,egalitarian,True,market,False,0.0
1,17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc,gaia,gpt-4o-mini,I’m researching species that became invasive a...,34689,"33139, 33140, 33141, 33142, 33143",False,"33139, 00601",False,2.0
2,04a04a9b-226c-43fd-b319-d5e89743676f,gaia,gpt-4o-mini,If we assume all articles published by Nature ...,41,"1,000",False,50,False,2.0
3,14569e28-c88c-43e4-8c32-097d35b9a67d,gaia,gpt-4o-mini,"In Unlambda, what exact charcter or text needs...",backtick,space,False,"Therefore, the character needed is a space",False,2.0
4,e1fc63a2-da7a-432f-be78-7c4a95598703,gaia,gpt-4o-mini,If Eliud Kipchoge could maintain his record-ma...,17,1000,False,1000,False,2.0


In [7]:
# Show only level 3 rows from final output
level_3_df = final_complexity_df[final_complexity_df["complexity_ground_truth"] == 3]
print("level 3 rows:", len(level_3_df))
level_3_df


level 3 rows: 4


,query_id,dataset,model,query,ground_truth,predicted_vanilla,is_correct_vanilla,predicted_cot,is_correct_cot,complexity_ground_truth
23,f0f46385-fc03-4599-b5d3-f56496c3e69f,gaia,gpt-4o-mini,In terms of geographical distance between capi...,"Indonesia, Myanmar","Brunei, Vietnam",False,"Brunei, Vietnam",False,3.0
62,56db2318-640f-477a-a82f-bc93ad13e882,gaia,gpt-4o-mini,The following numbers function similarly to IS...,"7, 9","3, 4",False,"2, 3",False,3.0
95,851e570a-e3de-4d84-bcfa-cc85578baa59,gaia,gpt-4o-mini,I thought we could try a fun word puzzle toget...,Briniest,repetition,False,BRAIN,False,3.0
109,c3a79cfe-8206-451f-aca8-3fec8ebe51d3,gaia,gpt-4o-mini,The year is 2022. I am at the National Air and...,8,3,False,3,False,3.0


In [2]:
import pandas as pd
df_gaia_vanilla = pd.read_parquet("../results2/unified_baseline/gaia/vanilla/gpt_4o_mini/responses.parquet")

In [3]:
df_gaia_vanilla

,query_id,dataset,modality,model,query,ground_truth,predicted_answer,is_correct,raw_model_output,reasoning_trace,...,completion_tokens,total_tokens,cost_usd,latency_s,error_type,error_message,system_prompt_version,user_prompt_version,prompt_hash,metadata
0,c61d22de-5f6c-4958-a7f6-5e9707bd3466,gaia,vanilla,gpt-4o-mini,A paper about AI regulation that was originall...,egalitarian,democratic,False,\boxed{democratic},\boxed{democratic},...,6,195,0.000032,1.118,None,None,v1,v1,bf1c3d3db80ab816db65a3c4ae6de77606827454016aec...,"{'file_name': '', 'level': '2'}"
1,17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc,gaia,vanilla,gpt-4o-mini,I’m researching species that became invasive a...,34689,"33139, 33140, 33141, 33142, 33143",False,"\boxed{33139, 33140, 33141, 33142, 33143}","\boxed{33139, 33140, 33141, 33142, 33143}",...,22,237,0.000045,1.020,None,None,v1,v1,558e758d60b818bfbb26df04d025a2ce6603ae3f06ce14...,"{'file_name': '', 'level': '2'}"
2,04a04a9b-226c-43fd-b319-d5e89743676f,gaia,vanilla,gpt-4o-mini,If we assume all articles published by Nature ...,41,1,False,\boxed{1},\boxed{1},...,5,195,0.000031,0.802,None,None,v1,v1,05edb417c18fa20e1134df202b37a68c951572066a792b...,"{'file_name': '', 'level': '2'}"
3,14569e28-c88c-43e4-8c32-097d35b9a67d,gaia,vanilla,gpt-4o-mini,"In Unlambda, what exact charcter or text needs...",backtick,space,False,\boxed{space},\boxed{space},...,5,210,0.000034,0.628,None,None,v1,v1,39b19c0779267a8a03a45e4b9c5cb997245f29568808ea...,"{'file_name': '', 'level': '2'}"
4,e1fc63a2-da7a-432f-be78-7c4a95598703,gaia,vanilla,gpt-4o-mini,If Eliud Kipchoge could maintain his record-ma...,17,2000,False,\boxed{2000},\boxed{2000},...,6,203,0.000033,0.656,None,None,v1,v1,7d7379cac8114c4af24a48950faa25ed4e66108464f088...,"{'file_name': '', 'level': '1'}"
5,32102e3e-d12a-4209-9163-7b3a104efe5d,gaia,vanilla,gpt-4o-mini,The attached spreadsheet shows the inventory f...,Time-Parking 2: Parallel Universe,The Dark Knight,False,\boxed{The Dark Knight},\boxed{The Dark Knight},...,7,167,0.000028,0.878,None,None,v1,v1,a7877697f8a5237aa06c5e3ee83ac65d8ae618ce3e7809...,{'file_name': '32102e3e-d12a-4209-9163-7b3a104...
6,8e867cd7-cff9-4e6c-867a-ff5ddc2550be,gaia,vanilla,gpt-4o-mini,How many studio albums were published by Merce...,3,3,True,\boxed{3},\boxed{3},...,5,158,0.000026,0.590,None,None,v1,v1,c10f6b19733ac46096b3f808592d00f64f17c9ecc34463...,"{'file_name': '', 'level': '1'}"
7,3627a8be-a77f-41bb-b807-7e1bd4c0ebdf,gaia,vanilla,gpt-4o-mini,The object in the British Museum's collection ...,142,40,False,\boxed{40},\boxed{40},...,5,193,0.000031,0.843,None,None,v1,v1,4798a937edb02335c37db9c980ac1213a411e115b82177...,"{'file_name': '', 'level': '2'}"
8,7619a514-5fa8-43ef-9143-83b66a43d7a4,gaia,vanilla,gpt-4o-mini,"According to github, when was Regression added...",04/15/18,01/12/20,False,\boxed{01/12/20},\boxed{01/12/20},...,9,155,0.000027,0.690,None,None,v1,v1,b345e688ffef7e18164f1923b7238d58a5f4e267ff97d7...,"{'file_name': '', 'level': '2'}"
9,ec09fa32-d03f-4bf8-84b0-1f16922c3ae4,gaia,vanilla,gpt-4o-mini,Here's a fun riddle that I think you'll enjoy....,3,2,False,\boxed{2},\boxed{2},...,5,572,0.000088,0.658,None,None,v1,v1,749bdeb09c740c956dcc83056b469237af7f4225741cb7...,"{'file_name': '', 'level': '1'}"


In [15]:
df_complexity_gaia = pd.read_parquet("../datasets/complexity/gaia/gaia_complexity_subset.parquet")

In [17]:
df_complexity_gaia.head()

,query,actual_answer,level,complexity,complexity_level,file_name,c_S,c_R,c_T,c_D,c_TT
0,A paper about AI regulation that was originall...,egalitarian,2,0.303,3,,0.0714,0.1158,0.05,0.0663,0.000
1,I’m researching species that became invasive a...,34689,2,0.256,2,,0.0672,0.1043,0.05,0.0350,0.000
2,If we assume all articles published by Nature ...,41,2,0.392,3,,0.0525,0.1981,0.00,0.0350,0.106
3,"In Unlambda, what exact charcter or text needs...",backtick,2,0.209,1,,0.0315,0.0826,0.05,0.0000,0.045
4,If Eliud Kipchoge could maintain his record-ma...,17,1,0.391,3,,0.0682,0.1431,0.10,0.0350,0.045


In [55]:
import numpy as np

scores = df_complexity_gaia["complexity"].values

# 3 thresholds for 4 classes
t1 = np.percentile(scores, 15)   # 0 → 1 boundary
t2 = np.percentile(scores, 45)   # 1 → 2 boundary
t3 = np.percentile(scores, 80)   # 2 → 3 boundary

print(f"T1 (0/1 boundary) : {t1:.5f}")
print(f"T2 (1/2 boundary) : {t2:.5f}")
print(f"T3 (2/3 boundary) : {t3:.5f}")

T1 (0/1 boundary) : 0.14760
T2 (1/2 boundary) : 0.21060
T3 (2/3 boundary) : 0.30640


In [56]:
def assign_level(score, t1, t2, t3):
    if   score < t1: return 0
    elif score < t2: return 1
    elif score < t3: return 2
    else:            return 3

df_complexity_gaia["complexity_level"] = df_complexity_gaia["complexity"].apply(
    lambda s: assign_level(s, t1, t2, t3)
)

print(df_complexity_gaia[["query", "complexity", "complexity_level"]].head(10))

                                               query  complexity  \
0  A paper about AI regulation that was originall...       0.303   
1  I’m researching species that became invasive a...       0.256   
2  If we assume all articles published by Nature ...       0.392   
3  In Unlambda, what exact charcter or text needs...       0.209   
4  If Eliud Kipchoge could maintain his record-ma...       0.391   
5  The attached spreadsheet shows the inventory f...       0.234   
6  How many studio albums were published by Merce...       0.358   
7  The object in the British Museum's collection ...       0.281   
8  According to github, when was Regression added...       0.257   
9  Here's a fun riddle that I think you'll enjoy....       0.199   

   complexity_level  
0                 2  
1                 2  
2                 3  
3                 1  
4                 3  
5                 2  
6                 3  
7                 2  
8                 2  
9                 1  


In [58]:
print("\nDistribution:")
print(df_complexity_gaia["complexity_level"].value_counts().sort_index())

print("\nScore range per class:")
print(df_complexity_gaia.groupby("complexity_level")["complexity"].agg(
    ["min", "max", "mean", "count"]
).round(4))

print("\nComplexity level vs GAIA level:")
print(pd.crosstab(
    df_complexity_gaia["level"],             # GAIA level 1,2,3
    df_complexity_gaia["complexity_level"]   # your 0,1,2,3
))


Distribution:
complexity_level
0    25
1    49
2    58
3    33
Name: count, dtype: int64

Score range per class:
                    min    max    mean  count
complexity_level                             
0                 0.042  0.147  0.1153     25
1                 0.148  0.209  0.1833     49
2                 0.211  0.305  0.2501     58
3                 0.312  0.528  0.3840     33

Complexity level vs GAIA level:
complexity_level   0   1   2   3
level                           
1                 12  15  18   8
2                 10  28  31  17
3                  3   6   9   8
